# Feature Engineering: Matchup

Признаки, описывающие **состав пары**:

- `hand_mix_team1`, `hand_mix_team2` - закодированная комбинация рук в паре: `RR` / `RL` / `LL`
- `team1_height_gap`, `team2_height_gap` - модуль разницы роста внутри пары
- `team1_age_gap`, `team2_age_gap` - модуль разницы возраста внутри пары

Результат: `data/features/matchup.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path("../../data/processed")
FEATURES = Path("../../data/features")
RAW = Path("../../data/raw")

df = pd.read_csv(PROCESSED / "matches.csv", parse_dates=["played_at"])
players = pd.read_csv(RAW / "men/players.csv").set_index("player_id")

def _hand_mix(h1, h2):
    hs = sorted([h for h in [h1, h2] if pd.notna(h)])
    if len(hs) < 2:
        return None
    return "".join(s[0].upper() for s in hs)  # RR, LR, LL


def team_features(df, players):
    records = []
    for _, row in df.iterrows():
        rec = {"match_id": row["match_id"]}
        for pid_cols, prefix in [(["player_id_1", "player_id_2"], "team1"),
                                  (["player_id_3", "player_id_4"], "team2")]:
            a, b = row[pid_cols[0]], row[pid_cols[1]]
            h_a = players.loc[a, "hand"] if a in players.index else np.nan
            h_b = players.loc[b, "hand"] if b in players.index else np.nan
            rec[f"{prefix}_hand_mix"] = _hand_mix(h_a, h_b)

            for attr in ["height", "age"]:
                v_a = players.loc[a, attr] if a in players.index else np.nan
                v_b = players.loc[b, attr] if b in players.index else np.nan
                rec[f"{prefix}_{attr}_gap"] = abs(v_a - v_b) if pd.notna(v_a) and pd.notna(v_b) else np.nan
        records.append(rec)
    return pd.DataFrame(records)

In [2]:
m = team_features(df, players)
print(f"Фич: {m.shape[1] - 1}")
print(m["team1_hand_mix"].value_counts().head(10))
print()
print(m[["team1_height_gap", "team1_age_gap"]].describe())

Фич: 8
team1_hand_mix  team1_side_mix   
RR              backhand-drive       1280
LR              backhand-drive        801
RR              backhand-backhand      62
                drive-drive            32
LR              drive-drive            28
Name: count, dtype: int64

       team1_height_gap  team1_age_gap
count       2476.000000    2532.000000
mean           7.377221       4.911532
std            5.704143       4.647095
min            0.000000       0.000000
25%            3.000000       2.000000
50%            6.000000       3.000000
75%           11.000000       7.000000
max           31.000000      30.000000


In [4]:
out_path = FEATURES / "matchup.csv"
m.to_csv(out_path, index=False)